# LSTM Prediction Map (2020)

Interactive choropleth map showing the LSTM gentrification predictions for Copenhagen neighbourhoods.

| Period | Source file |
|--------|-------------|
| 2020   | `results/models/lstm_predictions.csv` — column `prediction` |

> **Note:** The LSTM model was trained on 1990–2020 CVR business sequence data. No future projections are available (unlike XGBoost or Ensemble) as there are no future CVR sequences to run the model on directly.

**Prerequisites:** Run notebook 13 first to generate `lstm_predictions.csv`.

Map is saved to `results/figures/lstm_map_2020.html`.

## 1. Imports & Configuration

In [ ]:
import os
import json
import pandas as pd
import geopandas as gpd
import leafmap.foliumap as leafmap
import folium
from IPython.display import display, HTML

ROOT        = os.path.join('..', '..')
MODELS_DIR  = os.path.join(ROOT, 'results', 'models')
FIGS_DIR    = os.path.join(ROOT, 'results', 'figures')
SHP_PATH    = os.path.join(ROOT, 'data', 'raw', 'nabolag_inkl_y_kom_shapefile', 'cluster_outer_v1.shp')

os.makedirs(FIGS_DIR, exist_ok=True)

# Color scheme: 0 = Not Gentrified (red), 1 = Gentrified (green), NoPrediction = grey
COLOR_MAP = {
    0: '#d62728',   # red   – Not Gentrified
    1: '#2ca02c',   # green – Gentrified
    'NoPrediction': '#aec7e8',
}
MODEL_LABEL = 'LSTM'

print('Paths configured.')
print(f'Shapefile: {SHP_PATH}')
print(f'Models dir: {MODELS_DIR}')

## 2. Load Shapefile

In [ ]:
gdf_raw = gpd.read_file(SHP_PATH)
print(f'Shapefile loaded: {len(gdf_raw)} polygons  |  CRS: {gdf_raw.crs}')

# Detect join key
join_key = None
for cand in ['munic_clus', 'cluster_id', 'Cluster_id']:
    if cand in gdf_raw.columns:
        join_key = cand
        break
if join_key is None:
    raise ValueError('No suitable join key in shapefile. Expected: munic_clus, cluster_id, or Cluster_id.')

gdf_raw['cluster_id'] = gdf_raw[join_key].astype(str).str.strip().str.lstrip('0')
if gdf_raw.crs is None or gdf_raw.crs.to_epsg() != 4326:
    gdf_raw = gdf_raw.to_crs(epsg=4326)

print(f'Join key used: "{join_key}"')
print(f'Sample cluster IDs: {gdf_raw["cluster_id"].head(5).tolist()}')

## 3. Helper – Build Map

In [ ]:
def build_map(pred_df, year, save_path):
    """Merge predictions onto shapefile and return a leafmap.Map."""
    pred_df = pred_df.copy()
    pred_df['cluster_id'] = pred_df['cluster_id'].astype(str).str.strip().str.lstrip('0')

    merged = gdf_raw.merge(pred_df[['cluster_id', 'prediction']], on='cluster_id', how='left')
    n_matched = merged['prediction'].notna().sum()
    print(f'  {year}: {n_matched}/{len(merged)} polygons matched predictions')

    geojson_data = json.loads(merged.to_json())

    m = leafmap.Map(center=[55.6761, 12.5683], zoom=11)
    m.add_basemap('OpenStreetMap')
    m.add_basemap('CartoDB.Positron')

    def style_fn(feature):
        val = feature['properties'].get('prediction')
        try:
            key = int(val) if val is not None else 'NoPrediction'
        except (TypeError, ValueError):
            key = 'NoPrediction'
        color = COLOR_MAP.get(key, COLOR_MAP['NoPrediction'])
        return {
            'fillColor':   color,
            'color':       '#555555',
            'weight':      0.5,
            'fillOpacity': 0.7,
        }

    m.add_geojson(
        geojson_data,
        layer_name=f'{MODEL_LABEL} {year}',
        style_function=style_fn,
        info_mode='on_hover',
    )

    legend_html = (
        '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;'
        'background:white;padding:10px 14px;border-radius:8px;'
        'box-shadow:2px 2px 6px rgba(0,0,0,.3);font-size:13px;">'
        f'<b>{MODEL_LABEL} {year}</b><br>'
        '<i style="background:#2ca02c;width:14px;height:14px;display:inline-block;border-radius:2px;"></i>&nbsp;Gentrified<br>'
        '<i style="background:#d62728;width:14px;height:14px;display:inline-block;border-radius:2px;"></i>&nbsp;Not Gentrified<br>'
        '<i style="background:#aec7e8;width:14px;height:14px;display:inline-block;border-radius:2px;"></i>&nbsp;No Prediction'
        '</div>'
    )
    m.get_root().html.add_child(folium.Element(legend_html))

    m.to_html(save_path)
    print(f'  Saved → {save_path}')
    return m

print('Helper function defined.')

## 4. Map – 2020 (LSTM Predictions)

In [ ]:
df_lstm = pd.read_csv(os.path.join(MODELS_DIR, 'lstm_predictions.csv'))
print(f'LSTM predictions: {len(df_lstm)} rows  |  columns: {list(df_lstm.columns)}')
print(df_lstm['prediction'].value_counts().sort_index())

m2020 = build_map(df_lstm, 2020, os.path.join(FIGS_DIR, 'lstm_map_2020.html'))
display(m2020)

## 5. Open Map in Browser

In [ ]:
path = os.path.abspath(os.path.join(FIGS_DIR, 'lstm_map_2020.html'))
print(f'Map 2020: {path}')

# Uncomment to auto-open in the default browser
# import webbrowser
# webbrowser.open(path)